# 讓模型產生 tool call，再由應用層執行工具

這份教材的主題是工具呼叫的交接邊界：`ToolCallAction` 會把工具 schema 送進模型 API，模型回傳想呼叫的工具與參數；真正執行工具的仍然是你的應用層。

## 在 Colab 準備環境

如果你是在 Colab 開啟，先複製專案並切換到專案資料夾；如果你已經在專案資料夾，可以直接跳過這格。

In [ ]:
!git clone https://github.com/R300-AI/Agentic-SDK.git
%cd Agentic-SDK

## 定義模型可選擇的工具

這份 schema 會傳給 OpenAI-compatible 模型 API。模型不是直接執行工具，而是根據 schema 回傳工具名稱與 JSON 參數。

In [ ]:
import json
from types import SimpleNamespace
from unittest.mock import patch

from agentic_sdk import Workflow
from agentic_sdk.modules import PassThroughPerceive, ToolCallAction

tools = [
    {
        'type': 'function',
        'function': {
            'name': 'create_support_ticket',
            'description': '建立客服處理單。',
            'parameters': {
                'type': 'object',
                'properties': {
                    'title': {'type': 'string', 'description': '處理單標題'},
                    'priority': {'type': 'string', 'description': '優先度'},
                },
                'required': ['title', 'priority'],
                'additionalProperties': False,
            },
        },
    }
]

tools

## 準備真正會做事的函式

模型只負責提出要呼叫哪個工具與參數；這個函式代表你的應用層、後端 API 或資料庫操作。

In [ ]:
def create_support_ticket(title, priority):
    if priority not in {'low', 'normal', 'high'}:
        raise ValueError('priority 必須是 low、normal 或 high')
    return {'ticket_id': 'TCK-1001', 'title': title, 'priority': priority, 'status': 'created'}

## 用 mock 模型端點跑真正的 ToolCallAction

為了讓 notebook 不需要 API key，下面用一個 mock OpenAI-compatible client。它仍然走 `ToolCallAction` 的正式路徑：建立 action、把 tools 傳給模型端點、從模型回應取回 `tool_calls`。

In [ ]:
class FakeOpenAIClient:
    def __init__(self, *args, **kwargs):
        self.last_request = None
        self.chat = SimpleNamespace(
            completions=SimpleNamespace(create=self._create_completion)
        )

    def _create_completion(self, **kwargs):
        self.last_request = kwargs
        tool_call = SimpleNamespace(
            id='call_demo_001',
            type='function',
            function=SimpleNamespace(
                name='create_support_ticket',
                arguments=json.dumps(
                    {'title': '無法登入 AI Hub', 'priority': 'high'},
                    ensure_ascii=False,
                ),
            ),
        )
        return SimpleNamespace(
            choices=[SimpleNamespace(message=SimpleNamespace(content=None, tool_calls=[tool_call]))],
            usage=SimpleNamespace(prompt_tokens=72, completion_tokens=12),
            model=kwargs.get('model', 'mock-tool-model'),
        )

fake_client = FakeOpenAIClient()
with patch('agentic_sdk.llm.openai_compatible.OpenAI', return_value=fake_client):
    workflow = Workflow(
        workflow_name='Tool call demo Agent',
        perceive=PassThroughPerceive(),
        action=ToolCallAction(
            api_key='mock-key',
            base_url='https://mock.openai.local/v1',
            model='mock-tool-model',
            tools=tools,
            tool_choice='auto',
        ),
    )
    result = workflow.run('AI Hub 無法登入，請幫我開高優先度處理單。')

latest_tool_calls = result.entities.get('latest_tool_calls') or []
print(result.final_message)
print(latest_tool_calls)

## 確認 ToolCallAction 真的把 schema 送進模型 API

在正式環境中，這個 request 會送到你的 OpenAI-compatible 端點。這裡檢查 mock client 收到的參數，確認 `tools` 和 `tool_choice` 是由 `ToolCallAction` 傳出去的。

In [ ]:
print('model:', fake_client.last_request['model'])
print('tool_choice:', fake_client.last_request['tool_choice'])
print('tool names:', [tool['function']['name'] for tool in fake_client.last_request['tools']])

## 解析並檢查模型回傳的工具參數

`ToolCallAction` 只把模型想呼叫的工具整理成資料。應用層仍要檢查工具名稱是否允許、JSON 是否能解析、必要欄位是否存在。

In [ ]:
allowed_tools = {'create_support_ticket': create_support_ticket}
required_arguments = {'title', 'priority'}

call = latest_tool_calls[0]
function = call['function']
function_name = function['name']
arguments = json.loads(function['arguments'])

if function_name not in allowed_tools:
    raise ValueError(f'不允許的工具：{function_name}')
missing = required_arguments - set(arguments)
if missing:
    raise ValueError(f'缺少必要參數：{sorted(missing)}')

print(function_name)
print(arguments)

## 由應用層執行工具，正式環境再換成真模型端點

通過檢查後，才呼叫真正的函式。正式環境不是改 `latest_tool_calls`，而是把 mock client 換成真實的 OpenAI-compatible `api_key`、`base_url` 和支援工具呼叫的 `model`。

In [ ]:
tool_result = allowed_tools[function_name](**arguments)
print(tool_result)

# 正式環境改成真實端點；應用層的 whitelist、參數驗證與實際執行仍然保留。
# workflow = Workflow(
#     workflow_name='Production tool call Agent',
#     perceive=PassThroughPerceive(),
#     action=ToolCallAction(
#         api_key='<API key>',
#         base_url='<OpenAI-compatible base_url>',
#         model='<支援工具呼叫的模型>',
#         tools=tools,
#         tool_choice='auto',
#     ),
# )